# Bölüm kareleri: yapay zekâ karakteri, ücretsiz ekran kartında

Bu not defteri `npm run kaggle:paket` zip'indeki komutlardan **SDXL** ile bölümün 20 başlangıç
karesini çizer. Karakterin yüzü her karede aynı kalsın diye, seçtiğiniz portre **IP-Adapter**
ile referans olarak kullanılır.

**Ayarlar (sağdaki panel → Session options):** Accelerator **GPU T4 x2**, Internet **On**,
**Add Input** → komut zip'i (dataset).

**İki adımda çalışır:**
1. `PORTRE = None` iken **Run All** → 3 portre çizilir (aşağıda görünür, Output'ta `portreler/`).
   (Zip'te hazır bir `portre.png` varsa bu adımı atlayıp `PORTRE = "zip"` yazabilirsiniz.)
2. Beğendiğiniz portrenin numarasını `PORTRE = 2` gibi yazın ve tekrar **Run All** →
   aynı yüz yeniden çizilir ve 20 kare üretilir. Sonuç: `kareler-<dizi>-e<bölüm>.zip`
   ve `paket/` klasörü (klip-uret not defterine girdi olarak eklenebilir).

In [ ]:
# 1) Kurulum (yaklaşık 2 dakika)
# Kaggle'ın hazır torchao sürümü güncel diffusers ile çakışıyor (FqnToConfig hatası)
!pip uninstall -y -q torchao
!pip install -q -U "diffusers>=0.34" "transformers>=4.48" accelerate safetensors

In [ ]:
# 2) Ayarlar
import glob, json, os, shutil

PORTRE = None      # 1, 2 ya da 3 — portreyi seçtikten sonra yazın; zip'teki hazır portre için "zip"
SEEDS = [101, 202, 303]
WIDTH, HEIGHT = 1344, 768       # SDXL'in 16:9'a yakın çözünürlüğü
STEPS = 30
FACE_SCALE = 0.6                # yüz benzerliği: yükseltirseniz yüz daha çok benzer, sahne daha az uyar
ONLY = None                     # yalnızca bazı kareler: ["c03", "c04"]

plans = glob.glob("/kaggle/input/**/plan.json", recursive=True)
assert plans, "plan.json bulunamadı. Sağ üstte 'Add Input' ile komut zip'ini (dataset) ekleyin."
plan = json.load(open(plans[0]))
assert "characterLook" in plan, "Bu zip eski sürüm; 'npm run kaggle:paket' ile yeniden oluşturun."

WORK = "/kaggle/working"
PKG = f"{WORK}/paket"
os.makedirs(f"{PKG}/keyframes", exist_ok=True)
os.makedirs(f"{WORK}/portreler", exist_ok=True)
shutil.copy(plans[0], f"{PKG}/plan.json")
print(f"{plan['series']} · Bölüm {plan['episode']} · {len(plan['clips'])} sahne")

In [ ]:
# 3) Modeli yükle (ilk seferde ~7 GB indirir)
import torch
from diffusers import AutoencoderKL, StableDiffusionXLPipeline
from transformers import CLIPVisionModelWithProjection

# fp16'da siyah kare sorununu önleyen VAE
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
image_encoder = CLIPVisionModelWithProjection.from_pretrained(
    "h94/IP-Adapter", subfolder="models/image_encoder", torch_dtype=torch.float16
)
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    vae=vae,
    image_encoder=image_encoder,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
).to("cuda")
pipe.enable_vae_tiling()

NEGATIVE = (
    "smiling, grin, laughing, deformed face, asymmetric eyes, extra fingers, bad hands, blurry, "
    "lowres, watermark, text, logo, cartoon, anime, 3d render, plastic skin"
)
print("Hazır:", torch.cuda.get_device_name(0))

In [ ]:
# 4) Portreler: aynı tohumla her seferinde aynı yüz çıkar
from IPython.display import display

def portrait(seed):
    prompt = (
        f"Close-up character portrait of {plan['characterLook']}, neutral studio background, "
        f"soft cinematic light, looking at the camera, photorealistic, sharp focus, 85mm"
    )
    return pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE,
        width=1024,
        height=1024,
        num_inference_steps=STEPS,
        guidance_scale=6.0,
        generator=torch.Generator("cuda").manual_seed(seed),
    ).images[0]

if PORTRE is None:
    for i, seed in enumerate(SEEDS, 1):
        img = portrait(seed)
        img.save(f"{WORK}/portreler/portre-{i}.png")
        print(f"Portre {i}")
        display(img.resize((384, 384)))
    print("\nBeğendiğiniz portrenin numarasını yukarıdaki PORTRE ayarına yazıp tekrar Run All deyin.")
elif PORTRE == "zip":
    from diffusers.utils import load_image
    found = glob.glob("/kaggle/input/**/portre.png", recursive=True)
    assert found, "Zip'te portre.png yok. 'npm run kaggle:paket -- --portre=...' ile ekleyin."
    face = load_image(found[0])
    face.save(f"{PKG}/portre.png")
    print("Zip'teki portre kullanılıyor")
    display(face.resize((384, 384)))
else:
    face = portrait(SEEDS[PORTRE - 1])
    face.save(f"{PKG}/portre.png")
    print(f"Seçilen portre: {PORTRE}")
    display(face.resize((384, 384)))

In [ ]:
# 5) Kareler: karakterli karelerde portre referans olarak kullanılır
import time

if PORTRE is None:
    print("Önce portre seçin (PORTRE ayarı). Kareler çizilmedi.")
else:
    pipe.load_ip_adapter(
        "h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter-plus-face_sdxl_vit-h.safetensors"
    )
    done, failed = [], []
    for i, c in enumerate(plan["clips"]):
        key = c["key"]
        if ONLY and key not in ONLY:
            continue
        out = f"{PKG}/keyframes/{key}.png"
        if os.path.exists(out):
            print(f"✔ {key} zaten var, atlandı")
            continue
        t0 = time.time()
        try:
            # Karaktersiz karelerde referansın etkisi kapatılır
            pipe.set_ip_adapter_scale(FACE_SCALE if c["face"] else 0.0)
            img = pipe(
                prompt=c["keyframe"],
                negative_prompt=NEGATIVE,
                ip_adapter_image=face,
                width=WIDTH,
                height=HEIGHT,
                num_inference_steps=STEPS,
                guidance_scale=6.0,
                generator=torch.Generator("cuda").manual_seed(1000 + i),
            ).images[0]
            img.save(out)
            done.append(key)
            print(f"✔ {key}  {time.time() - t0:.0f} sn")
            display(img.resize((448, 256)))
        except Exception as e:
            failed.append(key)
            print(f"✖ {key}: {e}")
        torch.cuda.empty_cache()
    print(f"\nBiten: {len(done)} · Hatalı: {failed or 'yok'}")

In [ ]:
# 6) Kareleri zip'le
if PORTRE is not None:
    zip_base = f"{WORK}/kareler-{plan['series']}-e{plan['episode']}"
    shutil.make_archive(zip_base, "zip", PKG)
    print("🖼 İndirilecek dosya:", zip_base + ".zip")
    print("Hareketli video için: klip-uret not defterinde 'Add Input' → bu not defterinin çıktısı.")